# mini-llm-pt no Google Colab

Notebook para treinar o caminho BPE do projeto com checkpoints e metricas salvos no Google Drive.

Fluxo sugerido:
1. Ativar runtime com GPU no Colab.
2. Montar o Drive.
3. Clonar ou atualizar o repositorio.
4. Instalar dependencias.
5. Opcionalmente reconstruir dataset e tokenizer.
6. Treinar o Transformer BPE.
7. Gerar amostras e revisar metricas.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# Ajuste estas variaveis antes de rodar.
REPO_URL = ''  # Ex.: 'https://github.com/seu-usuario/mini-llm-pt.git'
REPO_BRANCH = 'main'
WORKSPACE_ROOT = Path('/content/drive/MyDrive/projetos')
PROJECT_NAME = 'mini-llm-pt'
PROJECT_DIR = WORKSPACE_ROOT / PROJECT_NAME

# Nome curto do experimento; sera usado para checkpoint e metricas.
EXP_NAME = 'colab_bpe_block128_gpu'

# Controle do pipeline.
REBUILD_DATASET = False
RETRAIN_TOKENIZER = False
RUN_QUALITY_CHECK = True
RESUME_FROM = ''  # Ex.: 'checkpoints/colab_bpe_block128_gpu.pt'

# Parametros do tokenizer BPE.
TOKENIZER_VOCAB_SIZE = 5000
TOKENIZER_MIN_FREQUENCY = 2
TOKENIZER_NUM_MERGES = 2000

# Parametros do treino BPE.
BLOCK_SIZE = 128
BATCH_SIZE = 12
EVAL_BATCH_SIZE = 12
EVAL_NUM_BATCHES = 20
MAX_ITERS = 15000
LEARNING_RATE = 8e-4
MIN_LEARNING_RATE = 8e-5
WARMUP_ITERS = 200
N_EMBD = 256
N_HEAD = 8
N_LAYER = 4
DROPOUT = 0.10
EVAL_INTERVAL = 500
PATIENCE = 6
GRAD_CLIP = 1.0
GRADIENT_ACCUMULATION_STEPS = 2
SEED = 42

PROJECT_DIR


In [ ]:
import os
import subprocess
import sys

WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

if not PROJECT_DIR.exists():
    if not REPO_URL:
        raise ValueError(
            'PROJECT_DIR nao existe no Drive e REPO_URL esta vazio. '
            'Preencha REPO_URL ou copie o projeto para o Drive.'
        )
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
else:
    print(f'Projeto ja encontrado em: {PROJECT_DIR}')
    if (PROJECT_DIR / '.git').exists():
        subprocess.run(['git', '-C', str(PROJECT_DIR), 'status', '--short'], check=True)

os.chdir(PROJECT_DIR)
print(f'Diretorio atual: {Path.cwd()}')


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipykernel'], check=True)

print('Dependencias instaladas.')


In [ ]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('GPU nao detectada; troque o runtime para GPU se quiser acelerar o treino.')


In [ ]:
import json
import shlex
import subprocess
import sys
from pathlib import Path


def run_command(command):
    print('\n>>>', ' '.join(shlex.quote(part) for part in command))
    subprocess.run(command, check=True)


def output_paths(exp_name):
    tokenizer_path = PROJECT_DIR / 'artifacts' / 'tokenizers' / 'bpe.json'
    checkpoint_path = PROJECT_DIR / 'checkpoints' / f'{exp_name}.pt'
    metrics_path = PROJECT_DIR / 'artifacts' / 'runs' / f'{exp_name}.jsonl'
    evaluation_path = PROJECT_DIR / 'artifacts' / 'evaluations' / f'{exp_name}_generation.txt'
    return tokenizer_path, checkpoint_path, metrics_path, evaluation_path


TOKENIZER_PATH, CHECKPOINT_PATH, METRICS_PATH, EVALUATION_PATH = output_paths(EXP_NAME)
print('Tokenizer:', TOKENIZER_PATH)
print('Checkpoint:', CHECKPOINT_PATH)
print('Metricas:', METRICS_PATH)
print('Avaliacao:', EVALUATION_PATH)


In [ ]:
if REBUILD_DATASET:
    run_command([sys.executable, '-m', 'scripts.process_wikipedia_raws'])
    run_command([sys.executable, '-m', 'scripts.build_splits'])

if RUN_QUALITY_CHECK:
    run_command([sys.executable, '-m', 'scripts.check_dataset_quality'])


In [ ]:
if RETRAIN_TOKENIZER:
    run_command(
        [
            sys.executable,
            '-m',
            'scripts.train_bpe_tokenizer',
            '--train-path',
            'data/splits/train.txt',
            '--tokenizer-path',
            str(TOKENIZER_PATH),
            '--num-merges',
            str(TOKENIZER_NUM_MERGES),
            '--vocab-size',
            str(TOKENIZER_VOCAB_SIZE),
            '--min-frequency',
            str(TOKENIZER_MIN_FREQUENCY),
        ]
    )
else:
    print(f'Usando tokenizer existente em: {TOKENIZER_PATH}')


In [ ]:
train_command = [
    sys.executable,
    '-m',
    'scripts.train_transformer_bpe',
    '--tokenizer-path',
    str(TOKENIZER_PATH),
    '--checkpoint-path',
    str(CHECKPOINT_PATH),
    '--metrics-path',
    str(METRICS_PATH),
    '--device',
    'cuda' if torch.cuda.is_available() else 'cpu',
    '--block-size',
    str(BLOCK_SIZE),
    '--batch-size',
    str(BATCH_SIZE),
    '--eval-batch-size',
    str(EVAL_BATCH_SIZE),
    '--eval-num-batches',
    str(EVAL_NUM_BATCHES),
    '--max-iters',
    str(MAX_ITERS),
    '--learning-rate',
    str(LEARNING_RATE),
    '--min-learning-rate',
    str(MIN_LEARNING_RATE),
    '--warmup-iters',
    str(WARMUP_ITERS),
    '--n-embd',
    str(N_EMBD),
    '--n-head',
    str(N_HEAD),
    '--n-layer',
    str(N_LAYER),
    '--dropout',
    str(DROPOUT),
    '--eval-interval',
    str(EVAL_INTERVAL),
    '--patience',
    str(PATIENCE),
    '--grad-clip',
    str(GRAD_CLIP),
    '--gradient-accumulation-steps',
    str(GRADIENT_ACCUMULATION_STEPS),
    '--seed',
    str(SEED),
]

if RESUME_FROM:
    train_command.extend(['--resume-from', RESUME_FROM])

run_command(train_command)


In [ ]:
run_command(
    [
        sys.executable,
        '-m',
        'scripts.evaluate_generation_bpe',
        '--checkpoint-path',
        str(CHECKPOINT_PATH),
        '--output-path',
        str(EVALUATION_PATH),
    ]
)

print('\nUltimas linhas do relatorio:')
print(EVALUATION_PATH.read_text(encoding='utf-8')[-2000:])


In [ ]:
import json

events = [json.loads(line) for line in METRICS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
print('Total de eventos:', len(events))
print('Ultimos 5 eventos:')
for event in events[-5:]:
    print(event)
